# 🎯 Module 05: Ensemble Methods

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 05 — Ensemble Methods  
> **Date:** 11 January 2026  
> **Previous:** 04 — Classification  
> **Next:** 06 — Clustering

---

**Ensemble methods** combine multiple models into one, producing stronger and more robust predictions than any single model alone.

In this notebook we'll cover:

- **Bagging** vs **Boosting** vs **Voting** vs **Stacking**
- **Random Forest**
- **Gradient Boosting**
- **XGBoost**, **LightGBM**, **CatBoost**
- **AdaBoost**
- **Voting** (hard vs soft)
- **Stacking** with a meta-model
- Why ensembles work — and when they don't

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what **ensemble learning** is and why it works
- Distinguish **bagging** from **boosting**
- Train and tune a **Random Forest**
- Understand **Gradient Boosting** at a high level
- Use **XGBoost**, **LightGBM**, **CatBoost**, and **AdaBoost**
- Build a **VotingClassifier** (hard & soft)
- Build a **StackingClassifier** with a meta-model
- Compare ensembles against a baseline
- Understand when ensembles can still overfit

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Load a real binary-classification dataset
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Classes:', data.target_names.tolist())

# A helper to evaluate any model
def evaluate(name, model, X_train, y_train, X_test, y_test, fit_time=None):
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    preds = model.predict(X_test)

    try:
        probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)
    except Exception:
        auc = float('nan')

    print(f'{name:<22} | '
          f'acc={accuracy_score(y_test, preds):.4f} | '
          f'f1={f1_score(y_test, preds):.4f} | '
          f'auc={auc:.4f} | '
          f'time={train_time:.2f}s')
    return model

print('Helper ready.')

---
## 🧠 1. What are Ensemble Methods?

**Ensemble Learning** combines predictions from multiple models to create a final prediction.

Instead of:

```text
Data → One Model → Prediction
```

we use:

```text
             ┌→ Model 1 ─┐
Data ────────┼→ Model 2 ─┼→ Combine → Final Prediction
             ├→ Model 3 ─┤
             └→ Model 4 ─┘
```

The individual models are called **base learners**.

> Several models can work together to reduce errors and improve generalization.

---
## 🧩 2. Main Ensemble Strategies

| Method | Main Idea |
| --- | --- |
| **Bagging** | Train models independently on different samples |
| **Boosting** | Train models sequentially to correct errors |
| **Voting** | Combine predictions from different models |
| **Stacking** | Use another model to combine model predictions |

### A simple mental model

```text
Bagging   → Many independent models
Boosting  → Models learn from previous mistakes
Voting    → Combine predictions
Stacking  → Learn HOW to combine predictions
```

---
## 🌳 3. Random Forest

**Random Forest** is a **bagging**-based ensemble of Decision Trees.

Instead of a single tree:

```text
              ┌→ Tree 1 ─┐
Data ─────────┼→ Tree 2 ─┼→ Vote/Average → Prediction
              ├→ Tree 3 ─┤
              └→ Tree N ─┘
```

Each tree gets:
- A **random bootstrap sample** of the training data
- A **random subset of features** considered at each split

### Classification

Trees **vote** → majority wins.

### Regression

Predictions are **averaged**.

### Important parameters

- `n_estimators` — number of trees
- `max_depth` — maximum tree depth
- `max_features` — features considered at each split
- `min_samples_split`, `min_samples_leaf`

In [ ]:
# Baseline: single Decision Tree
baseline_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
evaluate('Decision Tree', baseline_tree, X_train, y_train, X_test, y_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
evaluate('Random Forest (200)', rf, X_train, y_train, X_test, y_test)

In [ ]:
# Feature importances from a trained Random Forest
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=data.feature_names)
top = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
top[::-1].plot(kind='barh')
plt.title('Top 10 feature importances (Random Forest)')
plt.xlabel('importance')
plt.tight_layout(); plt.show()

In [ ]:
# Effect of n_estimators on accuracy
ns = [1, 5, 10, 25, 50, 100, 200, 400]
accs = []
for n in ns:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    accs.append(accuracy_score(y_test, rf_n.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(ns, accs, marker='o')
plt.xscale('log')
plt.xlabel('n_estimators (log scale)'); plt.ylabel('Test accuracy')
plt.title('Random Forest — effect of tree count')
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🚀 4. Gradient Boosting

Boosting works differently from Random Forest.

Instead of building independent trees, models are built **sequentially**:

```text
Model 1
   ↓
Find errors
   ↓
Model 2 focuses on errors
   ↓
Find remaining errors
   ↓
Model 3
   ↓
Final Ensemble
```

Each new model attempts to **improve the current ensemble**.

Gradient Boosting is usually built from **shallow Decision Trees** as weak learners.

In [ ]:
# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
)
evaluate('Gradient Boosting', gb, X_train, y_train, X_test, y_test)

---
## ⚡ 5. XGBoost

**XGBoost (Extreme Gradient Boosting)** is a highly optimized gradient-boosting algorithm.

### Why it became popular

- Strong predictive performance
- Regularization built in
- Efficient training
- Handles complex tabular relationships
- Handles missing values in many workflows
- Extensive hyperparameter control

### Key parameters

```text
n_estimators      → number of boosting rounds
max_depth         → max depth of each tree
learning_rate     → step size (shrinkage)
subsample         → row sampling per tree
colsample_bytree  → feature sampling per tree
```

⚠️ A very large number of trees + high complexity can cause **overfitting**.

In [ ]:
# XGBoost — install with: pip install xgboost
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    )
    evaluate('XGBoost', xgb, X_train, y_train, X_test, y_test)
except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')

---
## 💡 6. LightGBM

**LightGBM** is a gradient-boosting framework optimized for **speed** and **large tabular datasets**.

### What makes it different

- **Histogram-based** feature binning
- **Leaf-wise** tree growth (instead of level-wise)
- Very fast on large data

### Key parameters

```text
n_estimators       → number of boosting rounds
learning_rate      → step size
num_leaves         → max leaves per tree (KEY parameter)
max_depth          → max depth
min_child_samples  → min samples per leaf
```

⚠️ Leaf-wise growth is efficient but can **overfit** without constraints like `num_leaves`.

In [ ]:
# LightGBM — install with: pip install lightgbm
try:
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    evaluate('LightGBM', lgbm, X_train, y_train, X_test, y_test)
except ImportError:
    print('LightGBM not installed. Run: pip install lightgbm')

---
## 🐱 7. CatBoost

**CatBoost** is a gradient-boosting algorithm known for **excellent handling of categorical features**.

### Practical advantage

Strong support for categorical data **without manual one-hot encoding**.

### Key parameters

```text
iterations      → number of boosting rounds
learning_rate   → step size
depth           → tree depth
verbose         → print progress
```

In [ ]:
# CatBoost — install with: pip install catboost
try:
    from catboost import CatBoostClassifier
    cat = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        random_state=42,
        verbose=False,
    )
    evaluate('CatBoost', cat, X_train, y_train, X_test, y_test)
except ImportError:
    print('CatBoost not installed. Run: pip install catboost')

---
## 🎯 8. AdaBoost

**AdaBoost (Adaptive Boosting)** builds models sequentially while giving **more attention** to examples that previous models classified incorrectly.

```text
Initial Model
     ↓
Incorrect examples get more attention
     ↓
Next Model
     ↓
Repeat
```

A common base learner is a **shallow Decision Tree** (often a decision stump).

In [ ]:
# AdaBoost
ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.5,
    random_state=42,
)
evaluate('AdaBoost', ada, X_train, y_train, X_test, y_test)

---
## 🗳️ 9. Voting Classifier

Voting combines predictions from **different classification models**.

```text
Logistic Regression → Cat
SVM                 → Cat
Decision Tree       → Dog

Final → Cat
```

### Hard Voting

Uses **predicted classes** — simple majority vote.

### Soft Voting

Uses **predicted probabilities** and averages them.

> Soft voting often works better when the base models give **well-calibrated probabilities**.

In [ ]:
# Voting Classifier — hard vs soft
base_estimators = [
    ('lr', Pipeline([('sc', StandardScaler()),
                     ('lr', LogisticRegression(max_iter=200, random_state=42))])),
    ('svm', Pipeline([('sc', StandardScaler()),
                      ('svm', SVC(kernel='rbf', probability=True, random_state=42))])),
    ('tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
]

voting_hard = VotingClassifier(estimators=base_estimators, voting='hard')
evaluate('Voting (hard)', voting_hard, X_train, y_train, X_test, y_test)

voting_soft = VotingClassifier(estimators=base_estimators, voting='soft')
evaluate('Voting (soft)', voting_soft, X_train, y_train, X_test, y_test)

---
## 🧠 10. Stacking

**Stacking** combines different models using a **meta-model**.

```text
                ┌→ Logistic Regression ─┐
Input ──────────┼→ Random Forest ────────┼→ Meta Model → Final Prediction
                └→ SVM ─────────────────┘
```

### How it works

1. First-level models make predictions
2. A second model **learns how to combine** those predictions

### Trade-off

More flexible than simple voting — but also more complex and slower to train.

In [ ]:
# Stacking Classifier
stack = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ('svm', Pipeline([('sc', StandardScaler()),
                          ('svm', SVC(kernel='rbf', probability=True, random_state=42))])),
        ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=200),
    cv=5,
    n_jobs=-1,
)
evaluate('Stacking', stack, X_train, y_train, X_test, y_test)

---
## ⚖️ 11. Bagging vs Boosting

| | Bagging | Boosting |
| --- | --- | --- |
| Training | Mostly parallel / independent | Sequential |
| Main idea | Reduce variance | Correct previous errors |
| Example | Random Forest | XGBoost |
| Base models | Often deep trees | Often **weak** trees |
| Overfitting | Generally well controlled | Requires careful tuning |

### Easy way to remember

```text
Bagging   → Many independent models
Boosting  → Models learn from previous mistakes
```

---
## 🔥 12. Why Ensembles Work

Suppose three models make different mistakes:

```text
Model A → 90% correct
Model B → 89% correct
Model C → 91% correct
```

If their **errors are not perfectly correlated**, combining them can produce a **more robust** prediction.

### ⚠️ Important nuance

> "More models" ≠ "better."

The models need **useful diversity** and **appropriate training**.

---
## 📊 13. Full Comparison on the Same Dataset

In [ ]:
# Build and evaluate all major ensembles on the same split
results = []

def run(name, model):
    t0 = time.time()
    model.fit(X_train, y_train)
    dt = time.time() - t0
    preds = model.predict(X_test)
    try:
        probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)
    except Exception:
        auc = float('nan')
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'auc': auc,
        'time_s': dt,
    })

run('Decision Tree',    DecisionTreeClassifier(max_depth=5, random_state=42))
run('Random Forest',    RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
run('Gradient Boost',   GradientBoostingClassifier(n_estimators=200, random_state=42))
run('AdaBoost',         AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42))
run('Voting (soft)',    voting_soft)
run('Stacking',         StackingClassifier(
                            estimators=[
                                ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
                                ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
                            ],
                            final_estimator=LogisticRegression(max_iter=200),
                            cv=5, n_jobs=-1))

# Optional boosting libraries
try:
    from xgboost import XGBClassifier
    run('XGBoost', XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=4,
                                 random_state=42, eval_metric='logloss', n_jobs=-1))
except ImportError:
    pass
try:
    from lightgbm import LGBMClassifier
    run('LightGBM', LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=31,
                                   random_state=42, n_jobs=-1, verbose=-1))
except ImportError:
    pass
try:
    from catboost import CatBoostClassifier
    run('CatBoost', CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6,
                                       random_state=42, verbose=False))
except ImportError:
    pass

# Show as a table
df_results = pd.DataFrame(results).sort_values('accuracy', ascending=False)
df_results['accuracy'] = df_results['accuracy'].round(4)
df_results['f1']       = df_results['f1'].round(4)
df_results['auc']      = df_results['auc'].round(4)
df_results['time_s']   = df_results['time_s'].round(2)
print(df_results.to_string(index=False))

In [ ]:
# Bar chart of accuracies
plt.figure(figsize=(9, 4))
plt.barh(df_results['model'], df_results['accuracy'])
plt.xlabel('Test accuracy')
plt.title('Ensemble methods — accuracy comparison')
plt.xlim(0.9, 1.01)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

---
## 🧪 14. Practical Workflow

For a tabular classification problem:

```text
Dataset
   ↓
Preprocessing
   ↓
Train/Test Split
   ↓
Baseline Model
   ↓
Random Forest
   ↓
XGBoost / LightGBM / CatBoost
   ↓
Evaluate
   ↓
Hyperparameter Tuning
   ↓
Final Model
```

> **Always compare against a baseline** — never assume an ensemble is automatically better.

---
## 📝 15. Practice Checklist

Use a classification dataset and train each of these:

- [ ] Decision Tree (baseline)
- [ ] Random Forest
- [ ] AdaBoost
- [ ] XGBoost
- [ ] LightGBM
- [ ] CatBoost
- [ ] Voting Classifier
- [ ] Stacking Classifier

Compare:

- [ ] Accuracy
- [ ] Precision
- [ ] Recall
- [ ] F1
- [ ] ROC-AUC
- [ ] Training time

Also experiment with:

- [ ] `n_estimators`
- [ ] `learning_rate`
- [ ] `max_depth`

---
## 🏋️ 16. Hands-On Exercises

### Exercise 1 — Random Forest Tuning
Try `n_estimators ∈ {50, 100, 200, 500}` and `max_depth ∈ {3, 5, 10, None}`. Which combination is best?

### Exercise 2 — Gradient Boosting Learning Rate
Try `learning_rate ∈ {0.01, 0.05, 0.1, 0.3}`. How does accuracy change with `n_estimators`?

### Exercise 3 — XGBoost Regularization
Add `reg_alpha` and `reg_lambda` parameters. Do they help on this dataset?

### Exercise 4 — LightGBM Leaves
Try `num_leaves ∈ {15, 31, 63, 127}`. Watch for overfitting at high values.

### Exercise 5 — Voting Weights
Use `VotingClassifier(..., weights=[...])` to give more weight to the strongest model. Does it help?

### Exercise 6 — Stacking Meta-Model
Replace the final estimator with `RandomForestClassifier` or `GradientBoostingClassifier`. Compare.

### Exercise 7 — Baseline Sanity Check
Add a `DummyClassifier` as a baseline. How much does the best ensemble beat it by?

---
## 🎤 17. Interview Questions

1. **What is ensemble learning?**  
   Combining multiple models to produce a stronger, more robust prediction.

2. **Why do ensemble methods work?**  
   Diversity in the base models' errors can cancel out if the errors are not perfectly correlated.

3. **Bagging vs boosting?**  
   Bagging → parallel, independent models, reduces variance. Boosting → sequential, corrects prior errors, reduces bias.

4. **How does Random Forest work?**  
   Bagged decision trees using random samples and random feature subsets at each split.

5. **Why does Random Forest use random features?**  
   To **decorrelate** the trees so their errors don't all align.

6. **What is Gradient Boosting?**  
   Sequentially fitting weak learners to the **residual errors** of the current ensemble.

7. **XGBoost vs Random Forest?**  
   XGBoost is boosting (sequential, error-correcting); RF is bagging (parallel, variance-reducing).

8. **Main idea behind LightGBM?**  
   Histogram-based, leaf-wise tree growth for speed and efficiency.

9. **Why is CatBoost useful for categorical data?**  
   It natively handles categorical features without manual one-hot encoding.

10. **How does AdaBoost work?**  
    Sequentially, giving more weight to examples that previous models misclassified.

11. **Hard voting vs soft voting?**  
    Hard uses predicted classes; soft uses predicted probabilities.

12. **What is stacking?**  
    Using a meta-model to learn how to combine base model predictions.

13. **What is a meta-model?**  
    The second-level model that combines first-level predictions in stacking.

14. **Can ensemble models overfit?**  
    Yes — especially boosting models with too many trees or too high learning rates.

---
## 🏁 18. Key Takeaways

```text
Ensemble Learning
       ↓
Multiple Models
       ↓
Better / More Robust Predictions
```

### Main algorithms

```text
Bagging
  └── Random Forest

Boosting
  ├── Gradient Boosting
  ├── XGBoost
  ├── LightGBM
  ├── CatBoost
  └── AdaBoost

Combination
  ├── Voting
  └── Stacking
```

### The big four rules

> **Bagging = independent models.**  
> **Boosting = sequential error correction.**  
> **Voting = combine predictions.**  
> **Stacking = learn how to combine predictions.**

### 🧭 Golden Rule

> **Ensembles aren't magic — they need diverse, reasonably good base learners and careful tuning. Always compare against a baseline.**

---

### 🔗 What's Next?

**Module 06 — Clustering**

We shift from **supervised** to **unsupervised** learning — grouping unlabeled data with K-Means, DBSCAN, Hierarchical Clustering, and Gaussian Mixture Models.